In [36]:
import requests

def _prebuilt_placeholder(tool_data):
    def _run(*args, **kwargs):
        return {
            "tool_type": "prebuilt",
            "tool_id": tool_data["id"],
            "message": "Prebuilt tool execution placeholder"
        }
    return _run

def _custom_function_placeholder(tool_data):
    def _run(*args, **kwargs):
        pass
    return _run

In [ ]:
def _custom_api_placeholder(tool_data):
    input_schema = tool_data.get("input_schema") or {}
    output_schema = tool_data.get("output_schema") or {}
    allowed_fields = set(input_schema.get("properties", {}).keys())
    
    print(input_schema)
    def _run(tool_input=None, **kwargs):
        payload = {}
        if isinstance(tool_input, dict):
            payload.update(tool_input)
        payload.update({k: v for k, v in kwargs.items() if k in allowed_fields})

        api_url = tool_data.get("api_url")
        method = (tool_data.get("api_request_type") or "GET").upper()
        custom_message = tool_data.get("custom_message")
        timeout = tool_data.get("timeout", 10)

        if not api_url:
            return {"status_code": None, "output": None, "custom_message": custom_message, "error": "Missing api_url"}

        try:
            if method == "GET":
                response = requests.get(api_url, params=payload, timeout=timeout)
            elif method == "POST":
                response = requests.post(api_url, json=payload, timeout=timeout)
            else:
                return {"status_code": None, "output": None, "custom_message": custom_message, "error": f"Unsupported method {method}"}

            try:
                raw_output = response.json()
            except ValueError:
                raw_output = response.text if response.text else None

            if isinstance(raw_output, dict) and "properties" in output_schema:
                allowed_outputs = output_schema["properties"].keys()
                shaped_output = {k: raw_output.get(k) for k in allowed_outputs}
            else:
                shaped_output = raw_output

            return {"status_code": response.status_code, "output": shaped_output, "custom_message": custom_message}

        except requests.RequestException as exc:
            return {"status_code": None, "output": None, "custom_message": custom_message, "error": str(exc)}

    return _run

In [48]:
from typing import List
from langchain_core.tools import Tool

from manager import ToolRegistryManager

def build_langchain_tools(tool_ids: List[str]) -> List[Tool]:
    """
    Given a list of tool IDs, return LangChain-compatible Tool objects.
    """
    manager = ToolRegistryManager()
    langchain_tools: List[Tool] = []

    for tool_id in tool_ids:
        tool_data = manager.get_tool(tool_id)
        if not tool_data:
            print("tool is missing")

        tool_type = tool_data.get("type")
        name = tool_data.get("name")
        description = tool_data.get("description")

        # --- Placeholder execution functions ---
        if tool_type == "prebuilt":
            func = _prebuilt_placeholder(tool_data)

        elif tool_type == "custom_function":
            func = _custom_function_placeholder(tool_data)

        elif tool_type == "custom_api":
            func = _custom_api_placeholder(tool_data)

        else:
            continue

        langchain_tools.append(
            Tool(
                name=name,
                description=description,
                func=func
            )
        )

    return langchain_tools

In [49]:
tool_ids = [
    "agent_8b8af64a-5063-4eb3-a00c-86c40e74ce43",
    "agent_84f2d4f0-97ad-457c-9f8b-6a70c8eb80af",
    "agent_f35d529c-5c1a-4bf5-9959-f501cbbdd9b3"
]

tools = build_langchain_tools(tool_ids)

{'type': 'object'}


In [50]:
response = tools[2].invoke({"user_id": "123"})
response

{'status_code': 200,
 'output': [{'id': 'agent_f35d529c-5c1a-4bf5-9959-f501cbbdd9b3',
   'type': 'custom_api',
   'name': 'get_user_orders',
   'description': 'Fetch orders',
   'input_schema': {'type': 'object'},
   'output_schema': {'type': 'object'},
   'custom_message': 'take only the agent id',
   'api_url': 'http://127.0.0.1:8000/v1/custom-api',
   'api_request_type': 'GET',
   'metadata': {'created_at': '2026-01-11T07:52:02.864233+00:00'}}],
 'custom_message': 'take only the order id and provide it to the user'}

In [118]:
from langchain_core.tools import StructuredTool

def user_info(user_id: int):
    try:
        if kwargs.get("user_id") == 123:
            return {"user_id": "123", "name": "John Doe", "email": "jogn@123.com"}
        if kwargs.get("user_id") == 456:
            return {"user_id": "456", "name": "Jane Smith", "email": "smith@123.com"}
        return {"error": "User not found"}
    except:
        if user_id == 123:
            return {"user_id": "123", "name": "John Doe", "email": "adfafad@123.com"}
        if user_id == 456:
            return {"user_id": "456", "name": "Jane Smith", "email": "fgsfdg@123/com"}
        return {"error": "User not found"}

tool1 = StructuredTool.from_function(
    func=user_info,
    name="user_info",
    description="Get user information using user id"
)

In [119]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(temperature=0)

agent = create_agent(model=llm, tools=[tool1])

In [120]:
rahul=agent.invoke({"messages": [("user", "Get info for user 123")]})

In [121]:
rahul['messages'][-1].content

'Here is the information for user 123:\n- Name: John Doe\n- Email: adfafad@123.com'